
# NEMA simulation and OSEM

Simulate "measured" data from the NEMA/IEC body phantom with a known activity,
reconstruct it with SIRF OSEM and measure the contrast recovery of the hot
spheres.

The data are simulated by SIMIND and reconstructed with a SPECTUB model that
does not include the collimator blur or a scatter term, so OSEM treats
scattered events as if they came from the primary distribution. The model is
close to the physics that made the data, by construction, so the recovery
shown here is an optimistic best case.

<div class="alert alert-danger"><h4>Warning</h4>

This example takes about three to four minutes, nearly all of it in the SIMIND
run. With a collimator, only a tiny fraction of the simulated photons reach
the detector, and too few histories give speckled projections that do not
reconstruct. The run uses 8 MPI processes.

</div>

The phantom needs phantomgen:
``pip install "phantomgen @ git+https://github.com/varzakis/phantomgen"``.


## Setup
Check that SIMIND is installed and choose where the outputs are written.



In [ ]:
import os
import shutil
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import sirf.STIR as sirf

from simind_python_connector import SirfSimindAdaptor, configs
from simind_python_connector.builders import (
    STIRSPECTAcquisitionDataBuilder,
    STIRSPECTImageDataBuilder,
)
from simind_python_connector.converters.attenuation import get_attenuation_coefficient
from simind_python_connector.core.types import (
    DEFAULT_QUANTIZATION_SCALE,
    ScoringRoutine,
)
from simind_python_connector.phantoms import nema_iec_phantom


if shutil.which(os.environ.get("SIMIND_BIN") or "simind") is None:
    raise RuntimeError(
        "SIMIND executable not found ($SIMIND_BIN or simind on PATH). "
        "Install SIMIND before running this example."
    )

output_dir = Path(os.environ.get("SIMIND_EXAMPLES_OUTPUT", "output")) / "nema_osem"
output_dir.mkdir(parents=True, exist_ok=True)

GRID = (48, 48, 48)
VOXEL_MM = 7.0

## NEMA/IEC phantom
phantomgen's ``pet`` preset has an 8:1 sphere-to-background activity ratio.
Its two smallest spheres, 10 and 13 mm, are cold; the four hot spheres are
17, 22, 28 and 37 mm. The activity is in MBq per voxel. The attenuation map is
the density times the attenuation coefficient of water at 140 keV.



In [ ]:
nema = nema_iec_phantom(GRID, (VOXEL_MM,) * 3, preset="pet", supersample=4)
mu = nema.density_g_cm3 * get_attenuation_coefficient("water", 140.0)
slice_z = round(np.argwhere(nema.masks["sphere_6"]).mean(axis=0)[0])

fig, axes = plt.subplots(1, 2, figsize=(8, 4), constrained_layout=True)
for ax, (title, image) in zip(
    axes,
    (
        ("Activity (MBq per voxel)", nema.activity_mbq),
        ("Density (g/cm³)", nema.density_g_cm3),
    ),
):
    ax.imshow(image[slice_z], cmap="gray_r")
    ax.set_title(title)
    ax.axis("off")
plt.show()


def to_sirf_image(array):
    builder = STIRSPECTImageDataBuilder(
        {
            "!matrix size [1]": str(GRID[2]),
            "!matrix size [2]": str(GRID[1]),
            "!matrix size [3]": str(GRID[0]),
            "scaling factor (mm/pixel) [1]": str(VOXEL_MM),
            "scaling factor (mm/pixel) [2]": str(VOXEL_MM),
            "scaling factor (mm/pixel) [3]": str(VOXEL_MM),
        },
        backend="sirf",
    )
    builder.set_pixel_array(array.astype(np.float32))
    return builder.build()


activity_image = to_sirf_image(nema.activity_mbq)
mu_image = to_sirf_image(mu)
# STIR SPECTUB mirrors x relative to the mu map (UCL/STIR#452)
spectub_mu_image = to_sirf_image(mu[..., ::-1])

## Template
The template sets the projection geometry: 48 x 48 bins of 7 mm, 60 views
over 360 degrees and a detector radius of 250 mm. Both the simulation and the
reconstruction take their geometry from it.



In [ ]:
sirf.AcquisitionData.set_storage_scheme("memory")
template = STIRSPECTAcquisitionDataBuilder(
    header_overrides={
        "!matrix size [1]": "48",
        "!matrix size [2]": "48",
        "!number of projections": "60",
        "scaling factor (mm/pixel) [1]": str(VOXEL_MM),
        "scaling factor (mm/pixel) [2]": str(VOXEL_MM),
        "!direction of rotation": "CW",
        "start angle": "180",
        "Radius": "250",
    },
    backend="sirf",
).build(output_path=output_dir / "template")

## "Measured" data
SIMIND simulates the phantom for Tc-99m with a 126-154 keV window and the
ma-lehr collimator. The total activity and the time per view are given, so the
projections come out in counts (Index 25). The 8 MPI processes each simulate
all views and SIMIND sums them; without an ``RR`` switch their seeds are fixed.
Poisson noise from a seeded generator is added to the simulated counts.



In [ ]:
TOTAL_MBQ = float(nema.activity_mbq.sum())
TIME_PER_VIEW_S = 2.0

adaptor = SirfSimindAdaptor(
    configs.get("Example.yaml"),
    str(output_dir),
    "measured",
    scoring_routine=ScoringRoutine.PENETRATE,
    quantization_scale=0.06 * DEFAULT_QUANTIZATION_SCALE,
)
adaptor.set_template(template)
adaptor.set_source(activity_image)
adaptor.set_mu_map(mu_image)
# PENETRATE takes its energy window from Indexes 20 (upper) and 21 (lower).
adaptor.add_config_value(20, 154.0)
adaptor.add_config_value(21, 126.0)
adaptor.add_runtime_switch("CC", "ma-lehr")
adaptor.set_mpi(8)
adaptor.add_config_value(1, 140.0)
adaptor.set_activity(TOTAL_MBQ, TIME_PER_VIEW_S)
expected = adaptor.run()["all_interactions"]

rng = np.random.default_rng(12345)
measured = expected.clone()
measured.fill(rng.poisson(np.maximum(expected.as_array(), 0)).astype(np.float32))
print(f"Measured counts: {measured.sum():.4g}")

fig, axes = plt.subplots(1, 2, figsize=(8, 4), constrained_layout=True)
counts = measured.as_array()[0]  # axial, view, bin
axes[0].imshow(counts[:, 0, :], cmap="magma")
axes[0].set_title("Projection, view 0")
axes[1].imshow(counts[counts.shape[0] // 2], cmap="magma")
axes[1].set_title("Sinogram, central row")
axes[1].set_xlabel("bin")
axes[1].set_ylabel("view")
plt.show()

## OSEM
The model is SPECTUB with the flipped attenuation map and no PSF. OSEM runs
with 6 subsets of 10 views each, for 4 full iterations.



In [ ]:
acq_matrix = sirf.SPECTUBMatrix()
acq_matrix.set_attenuation_image(spectub_mu_image)
acq_matrix.set_resolution_model(0.0, 0.0, False)
acq_model = sirf.AcquisitionModelUsingMatrix(acq_matrix)

objective = sirf.make_Poisson_loglikelihood(measured)
objective.set_acquisition_model(acq_model)
objective.set_num_subsets(6)

recon = sirf.OSMAPOSLReconstructor()
recon.set_objective_function(objective)
recon.set_num_subsets(6)
recon.set_num_subiterations(24)
recon.set_input(measured)
reconstruction = activity_image.get_uniform_copy(1.0)
recon.set_up(reconstruction)
recon.reconstruct(reconstruction)
reconstruction.write(str(output_dir / "nema_osem.hv"))

## Reconstruction and truth
Both images are shown relative to their mean in the background mask, so the
arbitrary scale of the reconstruction does not matter.



In [ ]:
background = nema.masks["background"] > 0
truth_array = nema.activity_mbq / nema.activity_mbq[background].mean()
recon_array = reconstruction.as_array()
recon_array = recon_array / recon_array[background].mean()

fig, axes = plt.subplots(1, 2, figsize=(8, 4), constrained_layout=True)
for ax, (title, image) in zip(axes, (("Truth", truth_array), ("OSEM", recon_array))):
    shown = ax.imshow(image[slice_z], cmap="gray_r", vmin=0, vmax=truth_array.max())
    ax.set_title(title)
    ax.axis("off")
fig.colorbar(shown, ax=axes, label="relative to background")
plt.show()

## Contrast recovery
For each hot sphere, the mean in its mask over the mean in the background
mask, divided by the true ratio of 8. A value of 1 is full recovery. The
masks include voxels on the sphere edge, which hold less activity, so even
the truth scores below 1; its score is the ceiling for the reconstruction.
The 10 and 13 mm spheres are cold and are not scored. The simulation has a
modest number of photons, so the recovery is noisy and need not rise
smoothly with the sphere size.



In [ ]:
SPHERES = {"sphere_3": 17, "sphere_4": 22, "sphere_5": 28, "sphere_6": 37}
TRUE_RATIO = 8.0


def contrast_recovery(image, sphere_name):
    sphere = nema.masks[sphere_name] > 0
    return image[sphere].mean() / image[background].mean() / TRUE_RATIO


diameters = [str(d) for d in SPHERES.values()]
recovery = [contrast_recovery(recon_array, name) for name in SPHERES]
ceiling = [contrast_recovery(truth_array, name) for name in SPHERES]
for diameter, value, limit in zip(diameters, recovery, ceiling):
    print(f"{diameter} mm sphere: contrast recovery {value:.2f} (truth {limit:.2f})")

x = np.arange(len(diameters))
fig, ax = plt.subplots(figsize=(5, 4), constrained_layout=True)
ax.bar(x - 0.2, ceiling, 0.4, label="Truth")
ax.bar(x + 0.2, recovery, 0.4, label="OSEM")
ax.set_xticks(x, diameters)
ax.set_xlabel("Sphere diameter (mm)")
ax.set_ylabel("Contrast recovery")
ax.legend()
plt.show()